<a href="https://colab.research.google.com/github/Aleman-Mariana/Actividad-6--An-lisis-de-datos-con-sistemas-inteligentes/blob/main/Actividad_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Celda 1 - Instalar dependencia
!pip install fastmcp

In [ ]:
%%writefile database.py
import sqlite3

DB_NAME = "inventory.db"

def init_db():
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS productos (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            nombre TEXT NOT NULL,
            categoria TEXT,
            cantidad INTEGER NOT NULL,
            precio REAL NOT NULL
        )
    """)

    conn.commit()
    conn.close()

Overwriting database.py


In [ ]:
%%writefile server.py
from mcp.server.fastmcp import FastMCP
import sqlite3
from database import init_db, DB_NAME

init_db()

mcp = FastMCP("InventarioDB")

def get_connection():
    return sqlite3.connect(DB_NAME)

@mcp.tool()
def crear_producto(nombre: str, categoria: str, cantidad: int, precio: float) -> str:
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute(
        "INSERT INTO productos (nombre, categoria, cantidad, precio) VALUES (?, ?, ?, ?)",
        (nombre, categoria, cantidad, precio)
    )
    conn.commit()
    conn.close()
    return "Producto creado exitosamente"

@mcp.tool()
def consultar_producto(id: int) -> dict:
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM productos WHERE id = ?", (id,))
    row = cursor.fetchone()
    conn.close()
    if row:
        return {"id": row[0], "nombre": row[1], "categoria": row[2], "cantidad": row[3], "precio": row[4]}
    return {"error": "Producto no encontrado"}

@mcp.tool()
def actualizar_producto(id: int, cantidad: int) -> str:
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("UPDATE productos SET cantidad = ? WHERE id = ?", (cantidad, id))
    conn.commit()
    conn.close()
    return "Producto actualizado correctamente"

@mcp.tool()
def eliminar_producto(id: int) -> str:
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("DELETE FROM productos WHERE id = ?", (id,))
    conn.commit()
    conn.close()
    return "Producto eliminado correctamente"

@mcp.tool()
def listar_productos() -> list:
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM productos")
    rows = cursor.fetchall()
    conn.close()
    return [
        {"id": row[0], "nombre": row[1], "categoria": row[2], "cantidad": row[3], "precio": row[4]}
        for row in rows
    ]

@mcp.tool()
def calcular_valor_total_inventario() -> dict:
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT SUM(cantidad * precio) FROM productos")
    total = cursor.fetchone()[0]
    conn.close()
    return {"valor_total_inventario": total if total else 0}

@mcp.tool()
def productos_agotados() -> list:
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM productos WHERE cantidad = 0")
    rows = cursor.fetchall()
    conn.close()
    return [
        {"id": row[0], "nombre": row[1], "categoria": row[2], "cantidad": row[3], "precio": row[4]}
        for row in rows
    ]

@mcp.tool()
def producto_mas_costoso() -> dict:
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM productos ORDER BY precio DESC LIMIT 1")
    row = cursor.fetchone()
    conn.close()
    if row:
        return {"id": row[0], "nombre": row[1], "categoria": row[2], "cantidad": row[3], "precio": row[4]}
    return {"error": "No hay productos registrados"}

@mcp.tool()
def estadisticas_inventario() -> dict:
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT COUNT(*), AVG(cantidad), AVG(precio), SUM(cantidad * precio) FROM productos")
    total_productos, promedio_cantidad, promedio_precio, valor_total = cursor.fetchone()
    conn.close()
    return {
        "total_productos": total_productos,
        "promedio_cantidad": promedio_cantidad,
        "promedio_precio": promedio_precio,
        "valor_total": valor_total if valor_total else 0
    }

if __name__ == "__main__":
    mcp.serve()

Overwriting server.py


In [ ]:
%%writefile requirements.txt
fastmcp

Overwriting requirements.txt


In [ ]:
from database import init_db, DB_NAME
from server import (
    crear_producto, consultar_producto, actualizar_producto,
    eliminar_producto, listar_productos, calcular_valor_total_inventario,
    productos_agotados, producto_mas_costoso, estadisticas_inventario
)

print("✅ Todo importado correctamente")

✅ Todo importado correctamente


In [ ]:
print("=== CREANDO PRODUCTOS ===")
print(crear_producto("Laptop Lenovo", "Electrónica", 10, 2500000))
print(crear_producto("Mouse Inalámbrico", "Accesorios", 25, 85000))
print(crear_producto("Teclado Mecánico", "Accesorios", 0, 320000))
print(crear_producto("Monitor 24 pulgadas", "Electrónica", 5, 980000))
print(crear_producto("Disco Duro 1TB", "Almacenamiento", 0, 250000))

=== CREANDO PRODUCTOS ===
Producto creado exitosamente
Producto creado exitosamente
Producto creado exitosamente
Producto creado exitosamente
Producto creado exitosamente


In [ ]:
print("=== CONSULTAR PRODUCTO (ID=1) ===")
print(consultar_producto(1))

=== CONSULTAR PRODUCTO (ID=1) ===
{'id': 1, 'nombre': 'Laptop Lenovo', 'categoria': 'Electrónica', 'cantidad': 8, 'precio': 2500000.0}


In [ ]:
print("=== ACTUALIZAR PRODUCTO (ID=1, nueva cantidad=8) ===")
print(actualizar_producto(1, 8))
print("Verificando cambio:")
print(consultar_producto(1))

=== ACTUALIZAR PRODUCTO (ID=1, nueva cantidad=8) ===
Producto actualizado correctamente
Verificando cambio:
{'id': 1, 'nombre': 'Laptop Lenovo', 'categoria': 'Electrónica', 'cantidad': 8, 'precio': 2500000.0}


In [ ]:
print("=== ELIMINAR PRODUCTO (ID=5) ===")
print(eliminar_producto(5))

=== ELIMINAR PRODUCTO (ID=5) ===
Producto eliminado correctamente


In [ ]:
print("=== LISTAR TODOS LOS PRODUCTOS ===")
productos = listar_productos()
for p in productos:
    print(p)

=== LISTAR TODOS LOS PRODUCTOS ===
{'id': 1, 'nombre': 'Laptop Lenovo', 'categoria': 'Electrónica', 'cantidad': 8, 'precio': 2500000.0}
{'id': 2, 'nombre': 'Mouse Inalámbrico', 'categoria': 'Accesorios', 'cantidad': 25, 'precio': 85000.0}
{'id': 3, 'nombre': 'Teclado Mecánico', 'categoria': 'Accesorios', 'cantidad': 0, 'precio': 320000.0}
{'id': 4, 'nombre': 'Monitor 24 pulgadas', 'categoria': 'Electrónica', 'cantidad': 5, 'precio': 980000.0}
{'id': 6, 'nombre': 'Laptop Lenovo', 'categoria': 'Electrónica', 'cantidad': 10, 'precio': 2500000.0}
{'id': 7, 'nombre': 'Mouse Inalámbrico', 'categoria': 'Accesorios', 'cantidad': 25, 'precio': 85000.0}
{'id': 8, 'nombre': 'Teclado Mecánico', 'categoria': 'Accesorios', 'cantidad': 0, 'precio': 320000.0}
{'id': 9, 'nombre': 'Monitor 24 pulgadas', 'categoria': 'Electrónica', 'cantidad': 5, 'precio': 980000.0}
{'id': 10, 'nombre': 'Disco Duro 1TB', 'categoria': 'Almacenamiento', 'cantidad': 0, 'precio': 250000.0}


In [ ]:
print("=== VALOR TOTAL DEL INVENTARIO ===")
print(calcular_valor_total_inventario())

=== VALOR TOTAL DEL INVENTARIO ===
{'valor_total_inventario': 59050000.0}


In [ ]:
print("=== PRODUCTOS AGOTADOS ===")
agotados = productos_agotados()
for p in agotados:
    print(p)

=== PRODUCTOS AGOTADOS ===
{'id': 3, 'nombre': 'Teclado Mecánico', 'categoria': 'Accesorios', 'cantidad': 0, 'precio': 320000.0}
{'id': 8, 'nombre': 'Teclado Mecánico', 'categoria': 'Accesorios', 'cantidad': 0, 'precio': 320000.0}
{'id': 10, 'nombre': 'Disco Duro 1TB', 'categoria': 'Almacenamiento', 'cantidad': 0, 'precio': 250000.0}


In [ ]:
print("=== PRODUCTO MÁS COSTOSO ===")
print(producto_mas_costoso())

=== PRODUCTO MÁS COSTOSO ===
{'id': 1, 'nombre': 'Laptop Lenovo', 'categoria': 'Electrónica', 'cantidad': 8, 'precio': 2500000.0}


In [ ]:
print("=== ESTADÍSTICAS DEL INVENTARIO ===")
stats = estadisticas_inventario()
for clave, valor in stats.items():
    print(f"  {clave}: {valor}")

=== ESTADÍSTICAS DEL INVENTARIO ===
  total_productos: 9
  promedio_cantidad: 8.666666666666666
  promedio_precio: 891111.1111111111
  valor_total: 59050000.0


In [ ]:
%%writefile README.md
# MCP Inventory - Sistema de Gestión de Inventario

## Descripción
Servidor MCP desarrollado con FastMCP y SQLite que permite gestionar un inventario
mediante operaciones CRUD y consultas analíticas.

## Requisitos
- Python 3.8 o superior
- SQLite3 (incluido en Python)
- fastmcp

## Instalación
```bash
pip install fastmcp
```

## Ejecución del servidor
```bash
python server.py
```

## Ejecución de pruebas en Google Colab
Ejecutar las celdas del notebook en orden desde la celda 1 hasta la celda 14.

## Herramientas disponibles
- `crear_producto` — Agrega un nuevo producto al inventario
- `consultar_producto` — Consulta un producto por su ID
- `actualizar_producto` — Actualiza la cantidad de un producto
- `eliminar_producto` — Elimina un producto por su ID
- `listar_productos` — Lista todos los productos registrados
- `calcular_valor_total_inventario` — Calcula el valor total del inventario
- `productos_agotados` — Lista productos con cantidad 0
- `producto_mas_costoso` — Retorna el producto con mayor precio
- `estadisticas_inventario` — Muestra estadísticas generales del inventario

Overwriting README.md


In [ ]:
# Verificar que el servidor MCP está correctamente configurado
from mcp.server.fastmcp import FastMCP
from server import mcp

print("=== VERIFICACIÓN DEL SERVIDOR MCP ===")
print(f"Nombre del servidor: {mcp.name}")

# Listar todas las herramientas registradas
tools = [t for t in dir(mcp) if not t.startswith('_')]
print(f"\nServidor '{mcp.name}' inicializado correctamente ✅")
print(f"\nHerramientas registradas:")
herramientas = [
    "crear_producto", "consultar_producto", "actualizar_producto",
    "eliminar_producto", "listar_productos", "calcular_valor_total_inventario",
    "productos_agotados", "producto_mas_costoso", "estadisticas_inventario"
]
for h in herramientas:
    print(f"  ✅ {h}")

print("\nEl servidor está listo para recibir conexiones MCP.")

=== VERIFICACIÓN DEL SERVIDOR MCP ===
Nombre del servidor: InventarioDB

Servidor 'InventarioDB' inicializado correctamente ✅

Herramientas registradas:
  ✅ crear_producto
  ✅ consultar_producto
  ✅ actualizar_producto
  ✅ eliminar_producto
  ✅ listar_productos
  ✅ calcular_valor_total_inventario
  ✅ productos_agotados
  ✅ producto_mas_costoso
  ✅ estadisticas_inventario

El servidor está listo para recibir conexiones MCP.
